# Manejo de archivos CSV y JSON con clases propias

En este notebook se usan dos clases:

- `ManejadorCSV` (`manejador_csv.py`): importa con `csv.reader`, modifica y guarda con `csv.writer`, encabezados incluidos.
- `ManejadorJSON` (`manejador_json.py`): importa con `json.load`, modifica y guarda con `json.dump`.

Las modificaciones están implementadas dentro de las clases; el notebook solo las llama. Los archivos originales no se tocan: las salidas se guardan con nombres nuevos en `datos/modificados/`.

## Origen de los datos

Los archivos se descargaron manualmente desde Hugging Face y se guardaron sin modificar en `datos/originales/`.

| Formato | Archivo local | URL de origen |
|---|---|---|
| CSV | `datos/originales/ARCHIVO.csv` | https://huggingface.co/datasets/solomonk/reddit_mental_health_posts/blob/main/depression.csv |
| Json | datos/originales/ARCHIVO.json  | https://huggingface.co/datasets/mdplus/Datathon2024/tree/main/clindoc


## 1. Importar las clases

Se buscan los directorios padres hasta encontrar `src/clases_po` y se agrega `src` a `sys.path`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po.manejador_csv import ManejadorCSV
from clases_po.manejador_json import ManejadorJSON

## 2. Configurar rutas relativas

Todas las rutas son relativas a la carpeta del notebook (`src/sincronicas/`).

In [ ]:

CSV_ORIGINAL = Path("datos/originales/depression.csv")
JSON_ORIGINAL = Path("datos/originales/aci_train.jsonl")

CSV_MODIFICADO = Path("datos/modificados/depression_modificado.csv")
JSON_MODIFICADO = Path("datos/modificados/aci_train_modificado.json")

print("CSV original existe: ", CSV_ORIGINAL.exists())
print("JSON original existe:", JSON_ORIGINAL.exists())

---
# Parte A: CSV

## A1. Importar con `csv.reader`

`importar()` abre el archivo con `with open(...)`, lo lee con `csv.reader`, toma la primera fila como encabezados y guarda el resto como registros.

In [ ]:
csv_handler = ManejadorCSV(CSV_ORIGINAL)
csv_handler.importar()

print(csv_handler.info())
print()
csv_handler.mostrar(5)

## A2. Modificar los datos

Se aplican tres modificaciones, todas con métodos de la clase:

1. **Actualizar un valor:** se cambia la última columna del primer registro.
2. **Agregar un registro:** se añade una fila nueva al final.
3. **Eliminar un registro:** se elimina el segundo registro (índice 1).

In [ ]:
columna = csv_handler.encabezados[-1]   # ✏️ puedes elegir otra columna
anterior, nuevo = csv_handler.actualizar_valor(0, columna, "VALOR_ACTUALIZADO")
print(f"1) Fila 0, columna '{columna}': {anterior!r} -> {nuevo!r}")

# El nuevo registro copia la estructura de la primera fila y marca la última columna
nuevo_registro = dict(zip(csv_handler.encabezados, csv_handler.registros[0]))
nuevo_registro[columna] = "REGISTRO_AGREGADO"
csv_handler.agregar_registro(nuevo_registro)
print(f"2) Registro agregado al final: {csv_handler.registros[-1]}")

eliminado = csv_handler.eliminar_registro(1)
print(f"3) Registro eliminado (índice 1): {eliminado}")

print(f"\nRegistros después de modificar: {len(csv_handler.registros)}")

## A3. Guardar con `csv.writer`

`guardar()` escribe primero los encabezados y luego los registros en un archivo nuevo. Si la ruta de salida coincidiera con la del original, la clase lanza un error para protegerlo.

In [ ]:
ruta_csv = csv_handler.guardar(CSV_MODIFICADO)
print(f"CSV guardado en: {ruta_csv}")

# Verificación: se vuelve a leer el archivo nuevo con la misma clase
verificacion_csv = ManejadorCSV(ruta_csv)
verificacion_csv.importar()
print(verificacion_csv.info())
verificacion_csv.mostrar(3)
print("...")
print(",".join(verificacion_csv.registros[-1]))

---
# Parte B: JSON

## B1. Importar con `json.load`

`importar()` lee el archivo con `json.load`. Si el JSON es una lista de objetos la usa directamente; si es un objeto que contiene la lista (por ejemplo `{"data": [...]}`), la detecta automáticamente.

In [ ]:
json_handler = ManejadorJSON(JSON_ORIGINAL)
json_handler.importar()

print(json_handler.info())
print()
json_handler.mostrar(2)

## B2. Modificar los datos

1. **Actualizar un valor:** se cambia un campo del primer registro.
2. **Agregar un registro:** se añade un objeto nuevo al final.
3. **Eliminar un registro:** se elimina el segundo registro (índice 1).

In [ ]:
campo = json_handler.campos()[0]   # ✏️ puedes elegir otro campo
anterior, nuevo = json_handler.actualizar_valor(0, campo, "VALOR_ACTUALIZADO")
print(f"1) Registro 0, campo '{campo}': {anterior!r} -> {nuevo!r}")

nuevo_registro = dict(json_handler.registros[0])   # copia la estructura
nuevo_registro[campo] = "REGISTRO_AGREGADO"
json_handler.agregar_registro(nuevo_registro)
print(f"2) Registro agregado al final: {json_handler.registros[-1]}")

eliminado = json_handler.eliminar_registro(1)
print(f"3) Registro eliminado (índice 1): {eliminado}")

print(f"\nRegistros después de modificar: {len(json_handler.registros)}")

## B3. Guardar con `json.dump`

`guardar()` escribe el JSON completo en un archivo nuevo, con la misma estructura del original, `indent=2` y `ensure_ascii=False` para conservar tildes y caracteres especiales.

In [ ]:
ruta_json = json_handler.guardar(JSON_MODIFICADO)
print(f"JSON guardado en: {ruta_json}")

verificacion_json = ManejadorJSON(ruta_json)
verificacion_json.importar()
print(verificacion_json.info())
print("Último registro:", verificacion_json.registros[-1])

---
## Comprobación: los originales se conservan

Se vuelven a leer los originales para confirmar que no cambiaron.

In [ ]:
orig_csv = ManejadorCSV(CSV_ORIGINAL); orig_csv.importar()
orig_json = ManejadorJSON(JSON_ORIGINAL); orig_json.importar()

print(f"CSV  original: {len(orig_csv.registros)} registros | modificado: {len(verificacion_csv.registros)}")
print(f"JSON original: {len(orig_json.registros)} registros | modificado: {len(verificacion_json.registros)}")

## Resumen de modificaciones

| Formato | Actualizar valor | Agregar registro | Eliminar registro | Salida |
|---|---|---|---|---|
| CSV | Última columna de la fila 0 → `VALOR_ACTUALIZADO` | Fila nueva al final | Fila con índice 1 | `datos/modificados/..._modificado.csv` |
| JSON | Primer campo del registro 0 → `VALOR_ACTUALIZADO` | Objeto nuevo al final | Registro con índice 1 | `datos/modificados/..._modificado.json` |

Como se agrega un registro y se elimina otro, el total queda igual al original, pero el contenido cambió.